In [1]:
import argparse, time
import os, yaml, gc
import random
from tqdm import tqdm
import copy, json
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from rasterio.enums import Resampling
import geopandas as gpd
from glob import glob
import xarray, rioxarray, rasterio
from oggm import utils
import rioxarray
from rioxarray import merge
import holoviews as hv
import datashader as ds
import hvplot.xarray
from holoviews.operation.datashader import datashade, rasterize, inspect_points

In [3]:
folder = "iceboost_20251009"
PATH_IN = f"/media/maffe/nvme/iceboost_global_deploy/{folder}"
rgi, version = 7, '70G' # 62, 70G

## Inspect in this regions which projections are available and how many glaciers in each one

In [5]:
# This loop will display the unique epsg codes in each region and the number of glaciers for each epsg
unique_epsg = {}
for n, tif_file in enumerate(os.listdir(f'{PATH_IN}/RGI{version}/rgi{rgi}/')):

    # do not include Antarctic Peninsula files
    if not tif_file.endswith('.tif') or tif_file.startswith('AntPen_'):
        continue

    tif = rioxarray.open_rasterio(f'{PATH_IN}/RGI{version}/rgi{rgi}/{tif_file}').astype("float32")
    epsg = tif.rio.crs.to_epsg()

    if not epsg in unique_epsg.keys():
        unique_epsg[epsg] = []

    unique_epsg[epsg].append(tif)

for n, key in enumerate(unique_epsg.keys()):
    print(f"{n} EPSG {key} no. glaciers: {len(unique_epsg[key])}")

0 EPSG 32633 no. glaciers: 1411
1 EPSG 32629 no. glaciers: 83
2 EPSG 32635 no. glaciers: 172


## Now you can decide which epsg to use to create the complex product

In [10]:
my_epsg = 32635

# Create the glacier complex
complex_epsg = f"EPSG:{my_epsg}"

# These will be the new attributes of the complex. All the others will automatically iinherit from the first raster
complex_area = 0
complex_ground_truth_lats = []
complex_ground_truth_lons = []
complex_ground_truth_meas = []
complex_volume_ice = 0
complex_volume_error = 0
complex_volume_ice_bsl = 0
complex_volume_bsl_error = 0

complex_num_glaciers = 0
complex_list_tifs = []

#for n, tif_file in enumerate(os.listdir(f'{PATH_IN}/RGI{version}/rgi{rgi}/')):
for n, tif in enumerate(unique_epsg[my_epsg]):

    
    #if not tif_file.endswith('.tif'):
    #    continue
        
    #tif = rioxarray.open_rasterio(f'{PATH_IN}/RGI{version}/rgi{rgi}/{tif_file}').astype("float32")

    assert tif.rio.crs == complex_epsg, "Something wrong."

    #if not tif.rio.crs == complex_epsg:
    #    continue

    complex_num_glaciers += 1
    complex_list_tifs.append(tif)
    
    #print(f"{tif_file}\t{tif.attrs['volume']}")

    # These will be the new attributes of the complex tif file
    complex_area += tif.attrs['area']
    complex_ground_truth_lats += json.loads(tif.attrs['ground_truth_lats']) # list concatenation of lats
    complex_ground_truth_lons += json.loads(tif.attrs['ground_truth_lons']) # list concatenation of lons
    complex_ground_truth_meas += json.loads(tif.attrs['ground_truth_meas']) # list concatenation of ice thickness measurements
    complex_volume_ice += tif.attrs['volume']
    complex_volume_error += tif.attrs['volume_error']
    complex_volume_ice_bsl += tif.attrs['volume_bsl']
    complex_volume_bsl_error += tif.attrs['volume_bsl_error']

print(f"Total ice volume: {complex_volume_ice}")
print(f"Complex no. glaciers: {len(complex_list_tifs)}")

Total ice volume: 3033.7947331818423
Complex no. glaciers: 172


In [46]:
# Create the ESPG mosaics and save
save_epsg_mosaics = False

# Important here. I am merging with the max attribute and forcing the out resolution to 100x100m
mosaic_epsg = merge.merge_arrays(complex_list_tifs, method='max', nodata=np.nan, res=(100, 100))

print(f"Proj {my_epsg} we have {len(complex_list_tifs)} files. Merging Done.")

# Take care of the new attributes that need to be replaced
complex_name = f"{folder}_rgi{rgi}_v{version}_epsg_{my_epsg}"

mosaic_epsg.attrs['id'] = complex_name
mosaic_epsg.attrs['name'] = complex_name
mosaic_epsg.attrs['lat'] = np.nan
mosaic_epsg.attrs['lon'] = np.nan
mosaic_epsg.attrs['volume'] = complex_volume_ice
mosaic_epsg.attrs['volume_error'] = complex_volume_error
mosaic_epsg.attrs['volume_bsl'] = complex_volume_ice_bsl
mosaic_epsg.attrs['volume_bsl_error'] = complex_volume_bsl_error
mosaic_epsg.attrs['area'] = complex_area
mosaic_epsg.attrs['ground_truth_lats'] = json.dumps(complex_ground_truth_lats)
mosaic_epsg.attrs['ground_truth_lons'] = json.dumps(complex_ground_truth_lons)
mosaic_epsg.attrs['ground_truth_meas'] = json.dumps(complex_ground_truth_meas)

# save some RAM, potentially this does very little
del complex_list_tifs
gc.collect()

if save_epsg_mosaics:
        
        out = f"{PATH_IN}/RGI{version}/complex/rgi{rgi}/{complex_name}.tif" #_with_peninsula.tif
        mosaic_epsg.rio.to_raster(out, 
                                  #driver="COG",
                                  #compress="LZW",
                                  compress="deflate",
                                  dtype="float32",
                                 windowed=True) # this option is important to save RAM
        print(f'CRS: {mosaic_epsg.rio.crs}')
        print(f'Nodata: {mosaic_epsg.rio.nodata}')
        print(f'Resolution: {mosaic_epsg.rio.resolution()}')

        print(f'Saved: {out}')

Proj 32633 we have 1411 files. Merging Done.


## Test plot

In [47]:
plot_test = True
if plot_test:
    #test_tif = rioxarray.open_rasterio(f'/media/maffe/nvme/iceboost_global_deploy/iceboost_20251009/RGI62/complex/rgi19/iceboost_20251009_rgi19_v62_epsg_3031.tif')
    #test_tif = rioxarray.open_rasterio(f'/media/maffe/nvme/iceboost_global_deploy/iceboost_20251009/RGI62/iceboost_20251009_rgi19_v62_epsg3031.tif')
    #test_tif = rioxarray.open_rasterio(f'/media/maffe/nvme/iceboost_global_deploy/iceboost_20251009/RGI62/iceboost_20251009_rgi7_v62_epsg32633.tif')
    test_tif = rioxarray.open_rasterio(f'/media/maffe/nvme/iceboost_global_deploy/iceboost_20251009/RGI70G/complex/rgi7/iceboost_20251009_rgi7_v70G_epsg_32633.tif')
    #test_tif = rioxarray.open_rasterio(f'{out}')
    print(test_tif.rio.crs)
    print(test_tif.rio.nodata)
    print(test_tif.rio.resolution())
    print(test_tif.shape)
    #print(test_tif)
    
    
    # -------------------- hv plot --------------------
    plot_hv = test_tif.sel(band=1,
        #x=slice(-2.3e+6, -1.7e+6),
        #y=slice(8.6e+5, 3.4e+5),
    ).hvplot.image(x='x', y='y', rasterize=True, dynamic=True
    ).opts(
        cmap='turbo',               
        width=500,                 
        height=600,                 
        colorbar=True,
        xaxis=None,
        yaxis=None,
        colorbar_opts={'title': 'Thickness [m]',},
        colorbar_position='bottom',
    )

    from IPython.display import display
    display(plot_hv)

EPSG:32633
nan
(100.0, -100.0)
(5, 4340, 2309)


:DynamicMap   []
   :Image   [x,y]   (value)

In [48]:
## Check regional volume
print(f'We will check RGI{version} rgi{rgi} \n')
complex_volume_ice = 0
complex_volume_ice_error = 0
complex_area = 0

for n, tif_file in enumerate(os.listdir(f'{PATH_IN}/RGI{version}/complex/rgi{rgi}/')):

    tif = rioxarray.open_rasterio(f'{PATH_IN}/RGI{version}/complex/rgi{rgi}/{tif_file}')
    
    vol_ice_file = tif.attrs['volume']
    vol_ice_file_error = tif.attrs['volume_error']
    area_file = tif.attrs['area']
    
    complex_volume_ice += vol_ice_file
    complex_volume_ice_error += vol_ice_file_error
    complex_area += area_file

    print(f'{n} {tif_file} \t {vol_ice_file:.3f} pm {vol_ice_file_error:.3f}')

print('\n')
print(f'Volume region RGI{version} rgi{rgi} volume: {complex_volume_ice} {complex_volume_ice_error} area: {complex_area}')

We will check RGI70G rgi7 

0 iceboost_20251009_rgi7_v70G_epsg_32635.tif 	 3033.795 pm 627.472
1 iceboost_20251009_rgi7_v70G_epsg_32629.tif 	 10.534 pm 3.907
2 iceboost_20251009_rgi7_v70G_epsg_32633.tif 	 3687.087 pm 917.614


Volume region RGI70G rgi7 volume: 6731.415488835377 1548.9922051932974 area: 33958.83178921603
